In [22]:
#Trains and compares multiple classification models on the preprocessed dataset.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

CLEAN_PATH = "../data/heart_cleaned.csv"
PLOTS_DIR = "../output/heart_plots"
MODELS_DIR = "../output/heart_models"

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

In [23]:
# 1. LOAD PREPROCESSED DATA
df = pd.read_csv(CLEAN_PATH)
X = df.drop(columns=["target"])
y = df["target"]
print("X:",X)
print("Y:",y)

X:           age  sex  trestbps      chol  fbs   thalach  exang   oldpeak  ca  \
0   -0.267966    1 -0.377511 -0.704039    0  0.808993      0 -0.025091   2   
1   -0.157260    1  0.527318 -0.893880    1  0.237018      1  1.869266   0   
2    1.724733    1  0.828927 -1.505591    0 -1.082925      1  1.418229   0   
3    0.728383    1  1.009893 -0.893880    0  0.501006      0 -0.927166   1   
4    0.839089    0  0.406674  1.025627    1 -1.918889      0  0.786777   3   
..        ...  ...       ...       ...  ...       ...    ...       ...  ..   
297  1.503322    0 -0.679121 -0.725132    0 -1.522906      0  0.425947   0   
298 -1.153610    0 -1.402984 -2.201676    0  1.116980      0 -0.385921   0   
299 -0.267966    1 -0.196546  0.202981    0  0.501006      1 -0.927166   1   
300  0.506972    1  1.733756  0.582664    0 -1.082925      0 -0.927166   0   
301 -0.046555    1 -0.679121 -1.210282    0 -1.610902      0  0.335739   1   

      cp_1   cp_2   cp_3  restecg_1  restecg_2  slope_1  slo

In [24]:
# 2. TRAIN/TEST SPLIT (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

Train shape: (241, 19) | Test shape: (61, 19)


In [25]:
# 3. DEFINE MODELS
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes": GaussianNB(),
}

In [26]:
# 4. TRAIN, PREDICT & EVALUATE EACH MODEL
results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({
        "Model": name, "Accuracy": acc, "Precision": prec,
        "Recall": rec, "F1-Score": f1, "ROC-AUC": auc
    })
    trained_models[name] = model

    joblib.dump(model, f"{MODELS_DIR}/{name.replace(' ', '_').replace('(', '').replace(')', '')}.pkl")

    print(f"\n{name}")
    print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))


Logistic Regression
              precision    recall  f1-score   support

  No Disease       0.83      0.86      0.84        28
     Disease       0.88      0.85      0.86        33

    accuracy                           0.85        61
   macro avg       0.85      0.85      0.85        61
weighted avg       0.85      0.85      0.85        61


Decision Tree
              precision    recall  f1-score   support

  No Disease       0.76      0.68      0.72        28
     Disease       0.75      0.82      0.78        33

    accuracy                           0.75        61
   macro avg       0.76      0.75      0.75        61
weighted avg       0.75      0.75      0.75        61


Random Forest
              precision    recall  f1-score   support

  No Disease       0.76      0.79      0.77        28
     Disease       0.81      0.79      0.80        33

    accuracy                           0.79        61
   macro avg       0.79      0.79      0.79        61
weighted avg       0.79

In [28]:
# 5. MODEL COMPARISON TABLE
results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
print("\n=== Model Comparison ===")
print(results_df.round(4))
results_df.to_csv("../output/model_comparison.csv", index=False)


=== Model Comparison ===
                 Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
0  Logistic Regression    0.8525     0.8750  0.8485    0.8615   0.8983
1          Naive Bayes    0.8197     0.8438  0.8182    0.8308   0.8983
2            SVM (RBF)    0.8033     0.8182  0.8182    0.8182   0.8734
3                  KNN    0.8033     0.8621  0.7576    0.8065   0.8864
4        Random Forest    0.7869     0.8125  0.7879    0.8000   0.8939
5        Decision Tree    0.7541     0.7500  0.8182    0.7826   0.7484


In [13]:
# 6. VISUALIZE MODEL COMPARISON
plt.figure(figsize=(10, 6))
melted = results_df.melt(id_vars="Model", value_vars=["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"])
sns.barplot(data=melted, x="Model", y="value", hue="variable")
plt.xticks(rotation=20)
plt.ylim(0, 1.05)
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/model_comparison_bar.png")
plt.close()

In [14]:
# 7. CONFUSION MATRICES FOR ALL MODELS
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, model) in zip(axes.flat, trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["No Disease", "Disease"], yticklabels=["No Disease", "Disease"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/confusion_matrices.png")
plt.close()

In [17]:
# 8. FEATURE IMPORTANCE (Random Forest)
rf = trained_models["Random Forest"]
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8, 7))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/feature_importance_rf.png")
plt.close()
 
print("\nAll outputs saved: model_comparison.csv, plots, and trained model .pkl files")
print("\nBest model by Accuracy:", results_df.iloc[0]["Model"])


All outputs saved: model_comparison.csv, plots, and trained model .pkl files

Best model by Accuracy: Logistic Regression
